In [1]:
!pip install -q --upgrade transformers datasets evaluate sentencepiece psutil scikit-learn accelerate
print("Done!")

Done!


In [2]:
import transformers
print("Version:", transformers.__version__)  # should be 4.47+ or whatever latest is, that's fine

Version: 5.6.2


In [3]:
import os, time, json, gc
import numpy as np
import torch
import psutil
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAME = 'ai4bharat/indic-bert'
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS     = 3
OUTPUT_DIR = '/content/baseline_model'
os.makedirs('/content/results', exist_ok=True)
RESULTS = {}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — switch runtime!'}")

Device: cuda
GPU: Tesla T4


In [7]:
# Upload the CSV manually from Kaggle
# Step 1: Go to https://www.kaggle.com/datasets/praths71018/hindi-sentiment-dataset
# Step 2: Download the CSV to your computer
# Step 3: Run this cell and upload it when prompted

from google.colab import files
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

uploaded = files.upload()  # Click this and upload the CSV you downloaded from Kaggle

# Read whatever file was uploaded
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))
print("\nLabel distribution:")
print(df.iloc[:, -1].value_counts())

Saving output_combined_data_processed (1).csv to output_combined_data_processed (1).csv
Shape: (8000, 2)
Columns: ['Label', 'Sentence']

First 3 rows:
     Label                                           Sentence
0  neutral                             आप अपने हाथ भरे होंगे।
1  neutral                      कि मैंने किया। कि मैंने किया।
2  neutral  तो चलो अपने कर्तव्यों के बारे में थोड़ा बात करें।

Label distribution:
Sentence
ठीक है!                                                                              15
ओह!                                                                                  14
ठीक है?                                                                               9
क्या?                                                                                 9
अल्लाह!                                                                               9
                                                                                     ..
ओह भगवान, रोस मुझे बहुत खेद है।            

In [8]:
# Check what's actually in the file
print("Actual Label column unique values:")
print(df['Label'].unique()[:20])
print("\nTotal unique labels:", df['Label'].nunique())
print("\nValue counts:")
print(df['Label'].value_counts())

Actual Label column unique values:
['neutral' 'surprise' 'fear' 'sadness' 'joy' 'disgust' 'anger']

Total unique labels: 7

Value counts:
Label
neutral     3719
joy         1414
anger        930
surprise     899
sadness      560
fear         244
disgust      234
Name: count, dtype: int64


In [9]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

# ── Label mapping
label2id = {'neutral':0, 'joy':1, 'anger':2, 'surprise':3, 'sadness':4, 'fear':5, 'disgust':6}
id2label  = {v: k for k, v in label2id.items()}
NUM_LABELS = 7

# ── Clean dataframe
df = df[['Label', 'Sentence']].dropna()
df = df[df['Label'].isin(label2id.keys())]
df['label'] = df['Label'].map(label2id)
df = df[['Sentence', 'label']].reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Label distribution:")
print(df['label'].value_counts().sort_index())

# ── Train/test split (80/20, stratified so all 7 classes are in both splits)
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"\nTrain: {len(train_df)} | Test: {len(test_df)}")

Dataset shape: (8000, 2)
Label distribution:
label
0    3719
1    1414
2     930
3     899
4     560
5     244
6     234
Name: count, dtype: int64

Train: 6400 | Test: 1600


In [14]:
from transformers import AutoTokenizer

MODEL_NAME = 'ai4bharat/indic-bert'
MAX_LENGTH = 128

from huggingface_hub import login; login()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, keep_accents=True)

class HindiDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.texts  = df['Sentence'].tolist()
        self.labels = df['label'].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LENGTH,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = HindiDataset(train_df)
test_dataset  = HindiDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")
print("Tokenizer ready!")

config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

Train batches: 400
Test batches:  100
Tokenizer ready!


In [15]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
model = model.to(device)

total = sum(p.numel() for p in model.parameters())
print(f"Parameters:     {total:,}")
print(f"Estimated size: {total*4/1024**2:.1f} MB")
RESULTS['params'] = total

pytorch_model.bin:   0%|          | 0.00/135M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     | 
---------------------------------+------------+-
predictions.decoder.bias         | UNEXPECTED | 
sop_classifier.classifier.weight | UNEXPECTED | 
predictions.decoder.weight       | UNEXPECTED | 
predictions.LayerNorm.weight     | UNEXPECTED | 
sop_classifier.classifier.bias   | UNEXPECTED | 
predictions.dense.weight         | UNEXPECTED | 
predictions.LayerNorm.bias       | UNEXPECTED | 
predictions.bias                 | UNEXPECTED | 
predictions.dense.bias           | UNEXPECTED | 
classifier.bias                  | MISSING    | 
classifier.weight                | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters:     33,448,967
Estimated size: 127.6 MB


In [18]:
from torch.optim import AdamW
import torch.nn as nn

EPOCHS = 4

# ── Class weights (inverse frequency — penalises model for ignoring rare classes)
label_counts = torch.tensor([3719, 1414, 930, 899, 560, 244, 234], dtype=torch.float)
class_weights = (1.0 / label_counts)
class_weights = (class_weights / class_weights.sum() * NUM_LABELS).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

print("Class weights:", class_weights.cpu().numpy().round(3))

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = total_steps // 10

def get_lr(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)

best_f1, best_state = 0.0, None

for epoch in range(EPOCHS):
    model.train()
    total_loss, steps = 0, 0

    for batch in train_loader:
        batch  = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop('labels')
        out    = model(**batch)
        loss   = loss_fn(out.logits, labels)  # weighted loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        steps += 1

    model.eval()
    preds_all, labs_all = [], []
    with torch.no_grad():
        for batch in test_loader:
            labels = batch.pop('labels').to(device)
            batch  = {k: v.to(device) for k, v in batch.items()}
            preds  = model(**batch).logits.argmax(dim=-1).cpu().numpy()
            preds_all.extend(preds)
            labs_all.extend(labels.cpu().numpy())

    ep_f1 = f1_score(labs_all, preds_all, average='macro')
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/steps:.4f} | Macro F1: {ep_f1:.4f}")

    if ep_f1 > best_f1:
        best_f1 = ep_f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print(f"\nBest F1: {best_f1:.4f}")
RESULTS['baseline_f1'] = round(best_f1, 4)

Class weights: [0.141 0.372 0.565 0.585 0.938 2.154 2.246]
Epoch 1/4 | Loss: 1.7940 | Macro F1: 0.2503
Epoch 2/4 | Loss: 1.7114 | Macro F1: 0.2620
Epoch 3/4 | Loss: 1.5474 | Macro F1: 0.2954
Epoch 4/4 | Loss: 1.3654 | Macro F1: 0.3064

Best F1: 0.3064


In [19]:
import os, time, psutil

OUTPUT_DIR = '/content/baseline_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs('/content/results', exist_ok=True)

model.eval()

# ── Full classification report
preds_all, labs_all = [], []
with torch.no_grad():
    for batch in test_loader:
        labels = batch.pop('labels').to(device)
        batch  = {k: v.to(device) for k, v in batch.items()}
        preds  = model(**batch).logits.argmax(dim=-1).cpu().numpy()
        preds_all.extend(preds)
        labs_all.extend(labels.cpu().numpy())

print("Classification Report:")
print(classification_report(
    labs_all, preds_all,
    target_names=list(label2id.keys())
))

# ── Save model
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# ── Size
size_mb = sum(
    os.path.getsize(os.path.join(OUTPUT_DIR, f))
    for f in os.listdir(OUTPUT_DIR)
) / 1024**2
RESULTS['baseline_size_mb'] = round(size_mb, 1)
print(f"Model size: {size_mb:.1f} MB")

# ── Latency (100 runs)
sample = tokenizer(
    "यह बहुत अच्छा है।",
    return_tensors='pt', truncation=True,
    max_length=MAX_LENGTH, padding='max_length'
).to(device)

for _ in range(10):
    with torch.no_grad(): _ = model(**sample)

times = []
for _ in range(100):
    t = time.perf_counter()
    with torch.no_grad(): _ = model(**sample)
    times.append((time.perf_counter() - t) * 1000)

RESULTS['baseline_latency_ms'] = round(np.mean(times), 2)
print(f"Latency: {RESULTS['baseline_latency_ms']} ms/sample")

# ── RAM
proc = psutil.Process(os.getpid())
r0   = proc.memory_info().rss / 1024**2
with torch.no_grad(): _ = model(**sample)
RESULTS['baseline_ram_mb'] = round(proc.memory_info().rss/1024**2 - r0, 1)
print(f"RAM delta: {RESULTS['baseline_ram_mb']} MB")

# ── Save results + dataset (needed for later notebooks)
import json
with open('/content/results/all_results.json', 'w') as f:
    json.dump(RESULTS, f, indent=2)

# Save the splits so notebooks 2/3/4 use identical data
train_df.to_csv('/content/results/train.csv', index=False)
test_df.to_csv('/content/results/test.csv',  index=False)

print("\n=== BASELINE SUMMARY ===")
print(json.dumps(RESULTS, indent=2))

Classification Report:
              precision    recall  f1-score   support

     neutral       0.67      0.70      0.68       744
         joy       0.45      0.39      0.42       283
       anger       0.33      0.31      0.32       186
    surprise       0.37      0.40      0.38       179
     sadness       0.16      0.21      0.18       112
        fear       0.15      0.12      0.14        49
     disgust       0.04      0.02      0.03        47

    accuracy                           0.49      1600
   macro avg       0.31      0.31      0.31      1600
weighted avg       0.49      0.49      0.49      1600



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model size: 141.9 MB
Latency: 9.23 ms/sample
RAM delta: 0.0 MB

=== BASELINE SUMMARY ===
{
  "params": 33448967,
  "baseline_f1": 0.3064,
  "baseline_size_mb": 141.9,
  "baseline_latency_ms": 9.23,
  "baseline_ram_mb": 0.0
}


In [20]:
!zip -r /content/baseline_model.zip /content/baseline_model
!zip -r /content/results.zip /content/results

from google.colab import files
files.download('/content/baseline_model.zip')
files.download('/content/results.zip')
print("✅ Notebook 1 done! Save both zips — you need them for notebooks 2, 3, 4.")

  adding: content/baseline_model/ (stored 0%)
  adding: content/baseline_model/tokenizer.json (deflated 78%)
  adding: content/baseline_model/tokenizer_config.json (deflated 51%)
  adding: content/baseline_model/config.json (deflated 55%)
  adding: content/baseline_model/model.safetensors (deflated 7%)
  adding: content/results/ (stored 0%)
  adding: content/results/train.csv (deflated 79%)
  adding: content/results/all_results.json (deflated 34%)
  adding: content/results/test.csv (deflated 78%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Notebook 1 done! Save both zips — you need them for notebooks 2, 3, 4.
